# 02 -- Stage 2: DFDC Frame Extraction & Training (Kaggle)

Designed to run as a **Kaggle Notebook** with the DFDC dataset attached
via "Add Input" (no download/upload needed -- Kaggle mounts it read-only
under `/kaggle/input/...`).

Extracts frames using Kaggle's real DFDC format (a flat folder of `.mp4`
files plus `metadata.json` mapping `{"filename.mp4": {"label":
"REAL"|"FAKE", ...}}`), trains `ForensicMultiHeadNet` on them, and saves
`models/video_model.pth`.

**Run order note**: if you're running this *before* Stage 1 (no
`models/image_model.pth` yet), that's fine -- this notebook falls back to
randomly-initialized weights and trains standalone. Stage 3's merge step
reconciles image + video training regardless of which you ran first.

In [ ]:
import sys, os
PROJECT_ROOT = "/content/ForensicAI_Lab"  # adjust if you cloned/uploaded elsewhere
if not os.path.exists(PROJECT_ROOT):
    PROJECT_ROOT = os.getcwd()  # fallback: running locally from the project root
sys.path.insert(0, PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, install_requirements, is_colab

if is_colab():
    mount_drive()
    install_requirements(PROJECT_ROOT)
else:
    print("Not in Colab/Kaggle setup path -- assuming requirements-train.txt is already installed.")

In [ ]:
import torch, cv2, numpy as np, pandas as pd, json
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

from config.loader import get_settings, resolve_dataset_root, resolve_path
from services.video.extractor import extract_frames, get_video_fps, get_frame_count
from services.video.sampler import sample_frame_indices
from services.preprocessing.pipeline import preprocess_image
from services.detection.model import ForensicMultiHeadNet

settings = get_settings()
dataset_root = resolve_dataset_root(settings)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", device)

## On Kaggle: point `dataset_root` at the attached input

Kaggle attaches datasets read-only under `/kaggle/input/<dataset-slug>/`,
which won't match `dataset_root / "dfdc_subset"` automatically. Copy (or
symlink) it into place once per session -- adjust the slug to whatever
"Add Input" shows in your notebook's sidebar:

In [ ]:
import shutil

KAGGLE_INPUT_SLUG = "deepfake-detection-challenge"  # <-- change to your attached dataset's folder name
kaggle_input_path = Path("/kaggle/input") / KAGGLE_INPUT_SLUG

if kaggle_input_path.exists():
    dfdc_target = dataset_root / "dfdc_subset"
    if not dfdc_target.exists():
        shutil.copytree(kaggle_input_path, dfdc_target)
        print(f"Copied {kaggle_input_path} -> {dfdc_target}")
    else:
        print(f"{dfdc_target} already populated, skipping copy")
else:
    print("Not running on Kaggle with that input attached (or already prepared elsewhere) -- skipping.")

In [ ]:
# Kaggle's DFDC distribution is a flat folder of .mp4 files plus a
# metadata.json mapping filename -> {"label": "REAL"|"FAKE", "split": ...,
# "original": "<source.mp4>" (FAKE only)}. There may be several such
# folders if you downloaded multiple DFDC zip parts -- this picks up all
# metadata.json files found anywhere under dfdc_subset/.
dfdc_dir = dataset_root / "dfdc_subset"
metadata_files = list(dfdc_dir.glob("**/metadata.json"))
assert metadata_files, (
    f"No metadata.json found under {dfdc_dir} -- check the DFDC input/zip is in place."
)
print(f"Found {len(metadata_files)} metadata.json file(s)")

In [ ]:
frame_manifest_path = resolve_path("data/dfdc_frame_manifest.csv")
frames_out_dir = resolve_path("data/synthetic_data")
frames_out_dir.mkdir(parents=True, exist_ok=True)
rows = []

for meta_path in metadata_files:
    video_folder = meta_path.parent
    with open(meta_path) as f:
        metadata = json.load(f)

    for filename, info in metadata.items():
        video_path = video_folder / filename
        if not video_path.exists():
            continue

        label = "real" if info["label"].upper() == "REAL" else "fake"
        manip_type = "none" if label == "real" else "gan"  # deepfake/face-swap -> closest existing class

        fps = get_video_fps(str(video_path))
        total = get_frame_count(str(video_path))
        indices = set(sample_frame_indices(total, fps, settings.video.sample_fps, max_frames=10))

        for idx, frame in extract_frames(str(video_path)):
            if idx not in indices:
                continue
            frame_name = f"{video_path.stem}_frame{idx}.jpg"
            frame_path = frames_out_dir / frame_name
            cv2.imwrite(str(frame_path), frame)
            rows.append({"filename": str(frame_path), "label": label, "type": manip_type})

pd.DataFrame(rows, columns=["filename", "label", "type"]).to_csv(frame_manifest_path, index=False)
print("Wrote", len(rows), "frame rows to", frame_manifest_path)

## Train on the extracted frames

In [ ]:
class FrameDataset(Dataset):
    """Like Stage 1's image dataset, but reads full paths straight from the
    frame manifest (no images_dir join) and has no ground-truth masks --
    mask loss is zero-weighted for every sample via has_mask=0."""

    def __init__(self, manifest_csv, image_size):
        self.df = pd.read_csv(manifest_csv)
        self.image_size = image_size
        self.label_to_idx = {l: i for i, l in enumerate(settings.labels.binary)}
        manip_classes = settings.labels.manipulation_types
        self.manip_to_idx = {m: i for i, m in enumerate(manip_classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_bgr = cv2.imread(row["filename"])
        x = preprocess_image(image_bgr, tuple(self.image_size))

        label_idx = self.label_to_idx[row["label"]]
        manip_idx = self.manip_to_idx.get(row["type"], 0)
        mask = np.zeros((1, *self.image_size), dtype=np.float32)

        return (
            torch.from_numpy(x),
            torch.tensor(label_idx, dtype=torch.long),
            torch.tensor(manip_idx, dtype=torch.long),
            torch.from_numpy(mask),
            torch.tensor(0.0, dtype=torch.float32),
        )

In [ ]:
frame_dataset = FrameDataset(frame_manifest_path, settings.inference.image_size)
loader = DataLoader(frame_dataset, batch_size=16, shuffle=True, num_workers=2)
print("Frame dataset size:", len(frame_dataset))

In [ ]:
model = ForensicMultiHeadNet(
    backbone=settings.models.backbone,
    in_chans=settings.inference.input_channels,
    num_manip_classes=len(settings.labels.manipulation_types),
).to(device)

image_ckpt = resolve_path(settings.models.image_model_checkpoint)
if image_ckpt.exists():
    model.load_state_dict(torch.load(image_ckpt, map_location=device))
    print("Warm-started from Stage 1 checkpoint:", image_ckpt)
else:
    print("No Stage 1 checkpoint found -- training from randomly-initialized weights. "
          "That's fine; Stage 3's merge step reconciles image + video training either order.")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
cls_loss_fn = torch.nn.CrossEntropyLoss()
manip_loss_fn = torch.nn.CrossEntropyLoss()
mask_loss_fn = torch.nn.BCEWithLogitsLoss(reduction="none")

NUM_EPOCHS = 5  # adjust based on dataset size / Kaggle session limits

In [ ]:
model.train()
for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    for x, label_idx, manip_idx, mask, has_mask in loader:
        x, label_idx, manip_idx = x.to(device), label_idx.to(device), manip_idx.to(device)
        mask, has_mask = mask.to(device), has_mask.to(device)

        optimizer.zero_grad()
        out = model(x)

        loss_cls = cls_loss_fn(out["cls_logits"], label_idx)
        loss_manip = manip_loss_fn(out["manip_logits"], manip_idx)
        mask_loss_per_px = mask_loss_fn(out["mask_logits"], mask)
        loss_mask = (mask_loss_per_px.mean(dim=[1, 2, 3]) * has_mask).sum() / (has_mask.sum() + 1e-8)

        loss = loss_cls + loss_manip + loss_mask
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} -- loss: {running_loss / len(loader):.4f}")

In [ ]:
checkpoint_path = resolve_path(settings.models.video_model_checkpoint)
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), checkpoint_path)
print("Saved checkpoint to", checkpoint_path)

## Getting the checkpoint out of Kaggle

Kaggle keeps this session's output files under the notebook's "Output"
tab, but that's separate from your Google Drive. To use this checkpoint in
the Colab notebooks (Stage 3 merge, Stage 6 export), download
`models/video_model.pth` from the Output tab and upload it to
`ForensicAI_Lab/models/video_model.pth` on your Drive.